# Results for model: qwen/qwen3-next-80b-a3b-instruct

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np
from sklearn.model_selection import train_test_split

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Define rolling batch size (e.g., 1000 rows per batch)
batch_size = 1000
n_rows = len(df)
top_quantile_values = []

# Calculate global TOP_QUANTILE (top 15%) of feature_00 relative to responder_6 per batch
for i in range(0, n_rows, batch_size):
    batch = df[i:i+batch_size]
    if len(batch) == 0:
        continue
    # Sort by responder_6 descending to get highest responders
    sorted_batch = batch.sort('responder_6', descending=True)
    # Take top 15% of feature_00 values from this batch
    top_15_pct_idx = int(len(sorted_batch) * 0.15)
    if top_15_pct_idx > 0:
        top_15_feature_00 = sorted_batch['feature_00'][:top_15_pct_idx]
        global_top_quantile = top_15_feature_00.median()  # Use median as robust global quantile proxy
    else:
        global_top_quantile = batch['feature_00'].median()
    top_quantile_values.extend([global_top_quantile] * len(batch))

# Add global TOP_QUANTILE as new feature
df = df.with_columns(pl.Series(name='top_quantile_feature_00', values=top_quantile_values))

# Select features and target
feature_cols = [f'feature_{i:02d}' for i in range(130)] + ['top_quantile_feature_00']
X = df[feature_cols].to_numpy()
y = df['responder_6'].to_numpy()

# Remove rows with NaN in target
mask = ~np.isnan(y)
X = X[mask]
y = y[mask]

# Remove rows with NaN in features
mask_X = ~np.isnan(X).any(axis=1)
X = X[mask_X]
y = y[mask_X]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train XGBoost Regressor
model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Save model (optional, not required by output spec)
# model.save_model('xgb_model.json')